# 07 - Evaluation & Comparative Analysis

Evaluates RAG experiment results using RAGAS and DeepEval frameworks, then produces
comparative analysis tables and visualizations.

**Metrics:**
- RAGAS: Faithfulness, Answer Relevancy, Context Recall, Context Precision
- DeepEval: Contextual Precision, Contextual Recall, Faithfulness, Hallucination, Answer Relevancy

In [ ]:
import sys
sys.path.append("..")

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import ast
from datasets import Dataset
from langchain_huggingface import HuggingFaceEmbeddings

from ragas.metrics import faithfulness, answer_relevancy, context_recall, context_precision
from deepeval.metrics import (
    FaithfulnessMetric, ContextualRecallMetric,
    ContextualPrecisionMetric, AnswerRelevancyMetric
)

from src.llm_wrapper import get_groq_llm, GroqModel
from src.evaluation import evaluate_ragas, evaluate_deepeval, build_test_cases
import config

## Setup

In [ ]:
llm = get_groq_llm()
embeddings = HuggingFaceEmbeddings(model_name=config.EMBEDDING_MODELS[config.DEFAULT_EMBEDDING])
custom_llm = GroqModel(model=llm)

## Load Experiment Results

Load the RAG output CSVs generated in notebooks 05 and 06.

In [ ]:
import os

result_files = sorted(config.RESULTS_RAGAS_DIR.glob("*.csv"))
print("Available result files:")
for f in result_files:
    print(f"  {f.name}")

In [ ]:
# Load a specific experiment result for evaluation
# Update the filename to match your experiment
result_file = config.RESULTS_RAGAS_DIR / "naive_rag_minilm_chroma_cosine.csv"
eval_df = pd.read_csv(result_file)
eval_df["contexts"] = eval_df["contexts"].apply(ast.literal_eval)
eval_dataset = Dataset.from_pandas(eval_df)
print(f"Loaded {len(eval_dataset)} samples from {result_file.name}")

## RAGAS Evaluation

In [ ]:
# Context Recall
ragas_cr = evaluate_ragas(
    eval_dataset, context_recall, llm, embeddings,
    results_file=str(config.RESULTS_RAGAS_DIR / "eval_context_recall.csv")
)

In [ ]:
# Context Precision
ragas_cp = evaluate_ragas(
    eval_dataset, context_precision, llm, embeddings,
    results_file=str(config.RESULTS_RAGAS_DIR / "eval_context_precision.csv")
)

## DeepEval Evaluation

In [ ]:
test_cases = build_test_cases(eval_dataset)

ctx_precision_metric = ContextualPrecisionMetric(threshold=0.5, model=custom_llm)
ctx_recall_metric = ContextualRecallMetric(threshold=0.5, model=custom_llm)
faithfulness_metric = FaithfulnessMetric(model=custom_llm)
answer_rel_metric = AnswerRelevancyMetric(threshold=0.5, model=custom_llm)

In [ ]:
# DeepEval Context Recall
deepeval_cr = evaluate_deepeval(
    test_cases, ctx_recall_metric,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / "eval_ctx_recall_deepeval.csv")
)

In [ ]:
# DeepEval Context Precision
deepeval_cp = evaluate_deepeval(
    test_cases, ctx_precision_metric,
    results_file=str(config.RESULTS_DEEPEVAL_DIR / "eval_ctx_precision_deepeval.csv")
)

## Comparative Results Table

Consolidate all experiment results into a single comparison table.

In [ ]:
# Build comparison table from saved evaluation CSVs
# Update these with your actual result files
comparison = {
    "Experiment": [],
    "Vector DB": [],
    "Retrieval": [],
    "Embedding": [],
    "Context Recall": [],
    "Context Precision": [],
}

# Example row - replace with actual results
# comparison["Experiment"].append("Naive RAG")
# comparison["Vector DB"].append("ChromaDB")
# comparison["Retrieval"].append("Cosine")
# comparison["Embedding"].append("MiniLM")
# comparison["Context Recall"].append(0.864)
# comparison["Context Precision"].append(0.919)

# results_df = pd.DataFrame(comparison)
# results_df.to_csv(config.RESULTS_FIGURES_DIR / "comparison_table.csv", index=False)
# results_df

## Visualizations

In [ ]:
# Example: Bar chart comparing metrics across experiments
# Uncomment and populate with your results

# fig, axes = plt.subplots(1, 2, figsize=(14, 6))
#
# # Context Recall comparison
# axes[0].bar(results_df["Experiment"], results_df["Context Recall"], color="steelblue")
# axes[0].set_title("Context Recall by RAG Type")
# axes[0].set_ylabel("Score")
# axes[0].set_ylim(0, 1)
# axes[0].tick_params(axis="x", rotation=45)
#
# # Context Precision comparison
# axes[1].bar(results_df["Experiment"], results_df["Context Precision"], color="coral")
# axes[1].set_title("Context Precision by RAG Type")
# axes[1].set_ylabel("Score")
# axes[1].set_ylim(0, 1)
# axes[1].tick_params(axis="x", rotation=45)
#
# plt.tight_layout()
# plt.savefig(config.RESULTS_FIGURES_DIR / "metric_comparison.png", dpi=150, bbox_inches="tight")
# plt.show()